In [8]:
from pathlib import Path

import h5py
from archnemesis.helpers import h5py_helper

from archnemesis.database.filetypes.ans_line_data_file import AnsLineDataFile
from archnemesis.database.filetypes.ans_partition_fn_data_file import AnsPartitionFunctionDataFile



RECREATE_TEST_FILES = True

test_data_dir = Path("./test_data")


original_target_group_files = {
	'partition_function' : (test_data_dir / 'hitran24_pf.h5', '/sources/HITRAN24'),
	'line_data' : (test_data_dir / 'hitran24_lines.h5', '/sources/HITRAN24'),
	'pseudo_continuum' : (test_data_dir / 'hitran24_continuum.h5', '/sources/HITRAN24'),
}

result_file = test_data_dir / "hitran24_juipter_cirs_test_data.h5"

isos = [
	('C2H2',0),
	('C2H6',0),
	('CH4',1),
	('CH4',2),
	('CH4',3),
	('PH3',0),
	('NH3',0),
]


first_open = True
for target_group_name, (original_file, src_top_level_grp) in original_target_group_files.items():
	if original_file.exists():
		with h5py.File(original_file, 'r') as f:
			with h5py.File(result_file, 'w' if first_open else 'a') as g:
				first_open = False
				for mol_name, iso_id in isos:
					destination_grp = h5py_helper.ensure_grp(g, f'/{target_group_name}/{mol_name}') # NOTE: Location is one level above
					
					if iso_id == 0:
						i = 1
						grp_path = f'{src_top_level_grp}/{target_group_name}/{mol_name}/{i}'
						while grp_path in f:
							f.copy(grp_path, destination_grp)
							i += 1
							grp_path = f'{src_top_level_grp}/{target_group_name}/{mol_name}/{i}'
					else:
						grp_path = f'{src_top_level_grp}/{target_group_name}/{mol_name}/{iso_id}'
						if grp_path in f:
							f.copy(grp_path, destination_grp)
			
			
